# Notebook 11: Synthetic Data Generation & Constitutional AI

**Frontier AI Interview Prep** | Post-Training & Alignment Series

---

This notebook covers the methods that have become the backbone of modern alignment pipelines:
generating training data synthetically rather than relying solely on expensive human annotation.
We implement Self-Instruct, Evol-Instruct, Magpie, Constitutional AI, and RLAIF from scratch,
building toward a full synthetic preference data generator.

**Prerequisites**: Notebooks 01-10 (RLHF, DPO, Reward Modeling fundamentals).

**Runtime**: Google Colab GPU recommended. ~30 min with GPU, ~90 min CPU-only.

## 0. Self-Quiz (Active Recall)

Before reading anything, try to answer these from memory. Write your answers in the cell below,
then check them against the notebook content.

1. **What is Self-Instruct?** How does it bootstrap instruction-following data from a small seed set?
2. **What is Evol-Instruct?** What are its evolution operators and why do they produce harder examples?
3. **How does Constitutional AI work?** Walk through the three phases: generate, critique, revise.
4. **What is RLAIF?** When does AI feedback match or exceed human feedback quality?
5. **What is Magpie?** Why is it "prompt-free" and how does it exploit chat template structure?

---
*Tip: if you can explain each concept to a colleague without notes, you are interview-ready on this topic.*

In [ ]:
# YOUR ANSWERS (write before reading the notebook)
my_answers = {
    "self_instruct": "",
    "evol_instruct": "",
    "constitutional_ai": "",
    "rlaif": "",
    "magpie": "",
}
# After completing the notebook, come back and grade yourself.

## 1. Setup

In [ ]:
%%capture
!pip install torch transformers datasets openai accelerate sentencepiece protobuf

In [ ]:
import torch
import json
import random
import re
import copy
from collections import Counter
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import warnings
warnings.filterwarnings('ignore')

# Detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Set seed for reproducibility
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

## 2. Why Synthetic Data?

The economics of alignment are brutal:

| Data Source | Cost per Example | Speed | Consistency | Quality |
|---|---|---|---|---|
| Expert human annotation | $1-10 per preference pair | Slow (days-weeks) | Variable (inter-annotator disagreement) | High but noisy |
| Crowdsourced annotation | $0.10-1.00 | Medium | Low | Medium |
| Synthetic (strong model) | $0.001-0.01 | Fast (minutes-hours) | High (deterministic) | Surprisingly high |

### The Core Insight

**Use a strong model to generate training data for a weaker model.**

This is not cheating -- it is *distillation*. The strong model captures patterns that transfer
to the weaker model's parameter space. The key question is: when does this work, and when
does it fail?

### The Phi-1 Lesson: Data Quality > Data Quantity

Microsoft's Phi-1 ("Textbooks Are All You Need", Gunasekar et al. 2023) demonstrated that a
**1.3B parameter model** trained on ~7B tokens of high-quality synthetic "textbook" data could
match or exceed models 10x its size trained on 100x more web data on coding benchmarks.

The lesson: a carefully curated dataset of 1B tokens can outperform 100B tokens of noisy web
scrapes. This principle -- that **data quality dominates data quantity** -- is the foundation
of modern synthetic data pipelines.

### Why This Matters for Interviews

At frontier labs, synthetic data generation is no longer optional -- it is the primary way
alignment datasets are constructed. Understanding Self-Instruct, Evol-Instruct, Constitutional
AI, and RLAIF is table-stakes knowledge for any alignment researcher.

## 3. Self-Instruct

**Paper**: Wang et al. 2022, ["Self-Instruct: Aligning Language Models with Self-Generated Instructions"](https://arxiv.org/abs/2212.10560)

### The Algorithm

1. **Start with a seed set** of ~175 manually written (instruction, output) pairs
2. **Generate new instructions**: Prompt the model with a few seed examples, ask it to generate new instructions
3. **Classify** whether each instruction needs input or not
4. **Generate outputs**: Prompt the model with the instruction and (optionally) input to produce output
5. **Filter**: Remove duplicates, too-short, too-similar (ROUGE-L overlap), or degenerate outputs
6. **Repeat**: Add surviving examples to the pool, draw new samples, iterate

This was the method behind **Stanford Alpaca** -- they used GPT-3 (text-davinci-003) with Self-Instruct
to generate 52K instruction-following examples, then fine-tuned LLaMA 7B on them. The result was
remarkably competitive with GPT-3.5 on many tasks.

In [ ]:
class SelfInstruct:
    """Self-Instruct: bootstrap instruction-following data from seed examples.
    
    In production, this uses a strong API model (GPT-4, Claude). Here we
    simulate the pipeline structure with template-based generation to show
    the algorithm clearly, then demonstrate with a real model where available.
    """
    
    SEED_INSTRUCTIONS = [
        {
            "instruction": "Write a Python function that checks if a number is prime.",
            "input": "",
            "output": "def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True"
        },
        {
            "instruction": "Explain the difference between a stack and a queue.",
            "input": "",
            "output": "A stack follows Last-In-First-Out (LIFO) ordering: the most recently added element is removed first. A queue follows First-In-First-Out (FIFO) ordering: the element that has been in the queue the longest is removed first. Think of a stack as a pile of plates (you take from the top) and a queue as a line at a store (first person in line gets served first)."
        },
        {
            "instruction": "Summarize the following text in one sentence.",
            "input": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on the development of algorithms that can access data and use it to learn for themselves.",
            "output": "Machine learning is an AI subset where systems automatically learn and improve from data through algorithms rather than explicit programming."
        },
        {
            "instruction": "Convert the given temperature from Fahrenheit to Celsius.",
            "input": "98.6 degrees Fahrenheit",
            "output": "98.6 degrees Fahrenheit is equal to 37.0 degrees Celsius. Formula: (98.6 - 32) * 5/9 = 37.0"
        },
        {
            "instruction": "List three benefits of regular exercise.",
            "input": "",
            "output": "1. Improved cardiovascular health: Regular exercise strengthens the heart and improves blood circulation.\n2. Better mental health: Exercise releases endorphins that reduce stress, anxiety, and depression.\n3. Weight management: Physical activity helps burn calories and maintain a healthy body weight."
        },
        {
            "instruction": "Write a SQL query to find all employees with a salary greater than 50000.",
            "input": "",
            "output": "SELECT * FROM employees WHERE salary > 50000;"
        },
        {
            "instruction": "What is the time complexity of binary search?",
            "input": "",
            "output": "Binary search has a time complexity of O(log n), where n is the number of elements in the sorted array. At each step, it halves the search space by comparing the target with the middle element."
        },
        {
            "instruction": "Classify the sentiment of the following review as positive, negative, or neutral.",
            "input": "The food was absolutely amazing, but the service was incredibly slow. We waited 45 minutes for our appetizers.",
            "output": "Mixed/Neutral. The review expresses strong positive sentiment about the food quality but strong negative sentiment about the service speed."
        }
    ]
    
    # Prompt template for generating new instructions
    GENERATION_PROMPT = """Come up with a series of new tasks/instructions. These tasks should be diverse and cover different domains.

Here are some example tasks for reference:
{seed_examples}

Now generate {n_generate} new, diverse tasks. Each task should be a single clear instruction.
Format each as:
Task N: <instruction>
"""
    
    def __init__(self, generate_fn=None):
        """Initialize Self-Instruct.
        
        Args:
            generate_fn: A callable(prompt) -> str that generates text.
                         If None, uses a template-based simulator.
        """
        self.instruction_pool = list(self.SEED_INSTRUCTIONS)
        self.generate_fn = generate_fn or self._simulated_generate
        self.generated_data = []
    
    def _simulated_generate(self, prompt: str) -> str:
        """Simulated generation for demonstration purposes.
        Returns pre-crafted examples to show the pipeline structure.
        """
        new_tasks = [
            "Write a function to reverse a linked list in Python.",
            "Explain the concept of gradient descent in simple terms.",
            "Given a list of integers, find the two numbers that add up to a target sum.",
            "What are the ACID properties in database systems?",
            "Translate the following English sentence to formal language.",
            "Write a regular expression to validate email addresses.",
            "Describe three sorting algorithms and their time complexities.",
            "Explain the difference between supervised and unsupervised learning.",
            "Write a Python decorator that measures function execution time.",
            "What is the CAP theorem and why does it matter for distributed systems?",
            "Calculate the derivative of f(x) = 3x^2 + 2x - 5.",
            "Design a rate limiter for an API endpoint.",
        ]
        result = ""
        for i, task in enumerate(new_tasks, 1):
            result += f"Task {i}: {task}\n"
        return result
    
    def _simulated_output(self, instruction: str) -> str:
        """Generate a simulated output for a given instruction."""
        outputs = {
            "reverse a linked list": "def reverse_linked_list(head):\n    prev = None\n    current = head\n    while current:\n        next_node = current.next\n        current.next = prev\n        prev = current\n        current = next_node\n    return prev",
            "gradient descent": "Gradient descent is an optimization algorithm that iteratively adjusts parameters to minimize a loss function. Imagine you are on a mountain in fog -- you feel the slope under your feet and take a step downhill. You repeat this until you reach the valley (minimum). The 'learning rate' controls step size: too large and you overshoot, too small and you take forever.",
            "two numbers": "def two_sum(nums, target):\n    seen = {}\n    for i, num in enumerate(nums):\n        complement = target - num\n        if complement in seen:\n            return [seen[complement], i]\n        seen[num] = i\n    return []",
            "ACID": "ACID stands for:\n1. Atomicity: Transactions are all-or-nothing.\n2. Consistency: Transactions maintain database invariants.\n3. Isolation: Concurrent transactions don't interfere.\n4. Durability: Committed transactions survive crashes.",
        }
        for key, output in outputs.items():
            if key.lower() in instruction.lower():
                return output
        return f"[Generated output for: {instruction[:80]}...]"
    
    def generate_new_instructions(self, n_generate: int = 10, n_seed_examples: int = 4) -> List[str]:
        """Step 1: Generate new instructions from seed examples."""
        # Sample seed examples to include in prompt
        sampled = random.sample(self.instruction_pool, min(n_seed_examples, len(self.instruction_pool)))
        seed_text = "\n".join([f"- {ex['instruction']}" for ex in sampled])
        
        prompt = self.GENERATION_PROMPT.format(
            seed_examples=seed_text,
            n_generate=n_generate
        )
        
        raw_output = self.generate_fn(prompt)
        
        # Parse tasks from output
        instructions = []
        for line in raw_output.strip().split("\n"):
            match = re.match(r'Task \d+:\s*(.+)', line.strip())
            if match:
                instructions.append(match.group(1).strip())
        
        print(f"Generated {len(instructions)} new instructions")
        return instructions
    
    def filter_instructions(self, new_instructions: List[str],
                           min_length: int = 10,
                           max_overlap: float = 0.7) -> List[str]:
        """Step 2: Filter out low-quality or duplicate instructions.
        
        Filters:
        - Too short (< min_length chars)
        - Too similar to existing instructions (word overlap > max_overlap)
        - Exact duplicates
        """
        existing_instructions = {ex['instruction'].lower() for ex in self.instruction_pool}
        filtered = []
        
        for inst in new_instructions:
            # Length filter
            if len(inst) < min_length:
                continue
            
            # Exact duplicate filter
            if inst.lower() in existing_instructions:
                continue
            
            # Word overlap filter (simplified ROUGE-L proxy)
            inst_words = set(inst.lower().split())
            is_too_similar = False
            for existing in existing_instructions:
                existing_words = set(existing.split())
                if len(inst_words) == 0:
                    continue
                overlap = len(inst_words & existing_words) / len(inst_words)
                if overlap > max_overlap:
                    is_too_similar = True
                    break
            
            if not is_too_similar:
                filtered.append(inst)
                existing_instructions.add(inst.lower())
        
        print(f"Filtered: {len(new_instructions)} -> {len(filtered)} instructions")
        return filtered
    
    def generate_outputs(self, instructions: List[str]) -> List[Dict]:
        """Step 3: Generate outputs for each instruction."""
        results = []
        for inst in instructions:
            output = self._simulated_output(inst)
            results.append({
                "instruction": inst,
                "input": "",
                "output": output
            })
        print(f"Generated outputs for {len(results)} instructions")
        return results
    
    def run_iteration(self, n_generate: int = 10) -> List[Dict]:
        """Run one full Self-Instruct iteration."""
        print(f"\n{'='*60}")
        print(f"Self-Instruct Iteration (pool size: {len(self.instruction_pool)})")
        print(f"{'='*60}")
        
        # Step 1: Generate
        new_instructions = self.generate_new_instructions(n_generate)
        
        # Step 2: Filter
        filtered = self.filter_instructions(new_instructions)
        
        # Step 3: Generate outputs
        new_data = self.generate_outputs(filtered)
        
        # Add to pool
        self.instruction_pool.extend(new_data)
        self.generated_data.extend(new_data)
        
        print(f"Pool size: {len(self.instruction_pool)}")
        return new_data


# Run Self-Instruct for 3 iterations
si = SelfInstruct()
all_generated = []
for i in range(3):
    new_data = si.run_iteration(n_generate=10)
    all_generated.extend(new_data)

print(f"\n\nTotal generated: {len(all_generated)} instruction-output pairs")
print(f"\nSample generated pair:")
if all_generated:
    sample = all_generated[0]
    print(f"  Instruction: {sample['instruction']}")
    print(f"  Output: {sample['output'][:200]}...")

### Self-Instruct: Key Takeaways

**What the code shows**: The algorithm is simple -- sample from pool, generate new instructions,
filter bad ones, generate outputs, add to pool. The magic is in the filtering: removing duplicates
and near-duplicates prevents the pool from collapsing to a narrow distribution.

**In production**: Replace `_simulated_generate` with actual API calls to GPT-4 or Claude.
Stanford Alpaca used `text-davinci-003` (GPT-3.5) and generated 52K examples for ~$500.

**Limitations**:
- Diversity is bounded by the model's knowledge distribution
- Hard to generate truly novel instruction categories the model hasn't seen
- Quality degrades over iterations if filtering isn't aggressive enough

## 4. Evol-Instruct (WizardLM)

**Paper**: Xu et al. 2023, ["WizardLM: Empowering Large Language Models to Follow Complex Instructions"](https://arxiv.org/abs/2304.12244)

### The Insight

Self-Instruct generates diverse instructions, but they tend to be *simple*. Real users ask
complex, multi-step, constrained questions. Evol-Instruct addresses this by *evolving* simple
instructions into harder ones.

### Evolution Operators

| Operator | Description | Example |
|---|---|---|
| **Add Constraints** | Add limiting conditions | "Sort a list" -> "Sort a list of dictionaries by multiple keys, handling None values" |
| **Deepen** | Require deeper reasoning | "What is gradient descent?" -> "Derive the convergence rate of gradient descent for strongly convex functions" |
| **Concretize** | Make abstract concrete | "Write a sorting function" -> "Implement merge sort for a linked list with O(1) space" |
| **Increase Steps** | Require more reasoning steps | "Find the max" -> "Find the kth largest element without sorting, then prove its time complexity" |
| **Broaden** | Generalize the scope | "Sort integers" -> "Implement a generic sorting framework supporting any comparable type" |

In [ ]:
class EvolInstruct:
    """Evol-Instruct: evolve simple instructions into complex ones.
    
    The key idea: instead of generating instructions from scratch, take
    existing simple instructions and make them harder using specific
    evolution operators.
    """
    
    EVOLUTION_PROMPTS = {
        "add_constraints": """I want you to act as an Instruction Rewriter.
Given the following instruction, add 2-3 additional constraints or requirements
to make it more challenging and specific. The evolved instruction should be
reasonable and solvable.

Original Instruction: {instruction}

Evolved Instruction (with added constraints):""",

        "deepen": """I want you to act as an Instruction Rewriter.
Given the following instruction, rewrite it to require deeper domain knowledge
or more sophisticated reasoning. Increase the intellectual depth.

Original Instruction: {instruction}

Evolved Instruction (deeper reasoning required):""",

        "concretize": """I want you to act as an Instruction Rewriter.
Given the following instruction, make it more specific and concrete.
Replace general concepts with specific instances, add concrete parameters,
or specify exact implementation details.

Original Instruction: {instruction}

Evolved Instruction (more concrete and specific):""",

        "increase_steps": """I want you to act as an Instruction Rewriter.
Given the following instruction, rewrite it to require more intermediate
reasoning steps. The solution should involve a multi-step process.

Original Instruction: {instruction}

Evolved Instruction (more steps required):""",

        "broaden": """I want you to act as an Instruction Rewriter.
Given the following instruction, broaden its scope to cover a wider range
of scenarios or to generalize the problem.

Original Instruction: {instruction}

Evolved Instruction (broader scope):"""
    }
    
    # Pre-computed evolution examples for demonstration
    EVOLUTION_EXAMPLES = {
        "Write a Python function that checks if a number is prime.": {
            "add_constraints": "Write a Python function that checks if a number is prime, handling edge cases for negative numbers and floats. The function must run in O(sqrt(n)) time, use no external libraries, and return a descriptive error message for invalid inputs.",
            "deepen": "Implement the Miller-Rabin primality test in Python with configurable confidence levels. Explain why deterministic trial division is insufficient for cryptographic applications and analyze the probability of false positives for k rounds of testing.",
            "concretize": "Write a Python function that finds all prime numbers in the range [2, 10^6] using the Sieve of Eratosthenes, stores them in a sorted array, and supports O(log n) membership queries via binary search.",
            "increase_steps": "Write a Python function that: (1) checks if a number is prime using trial division, (2) if prime, finds the next prime after it, (3) computes the prime gap, and (4) verifies whether this gap size appears in the first 1000 prime gaps. Return all four results.",
            "broaden": "Implement a comprehensive primality testing framework in Python that supports multiple algorithms (trial division, Fermat's test, Miller-Rabin, and AKS), allows the user to select the algorithm, and benchmarks each approach for numbers of varying sizes."
        },
        "Explain the difference between a stack and a queue.": {
            "add_constraints": "Explain the difference between a stack and a queue, including their time complexities for all operations, at least two real-world applications for each, and describe how to implement a queue using two stacks with amortized O(1) dequeue.",
            "deepen": "Compare stacks and queues from a theoretical computer science perspective. Discuss their relationship to automata theory (PDA vs queue automata), prove that a queue automaton is strictly more powerful than a PDA, and explain implications for language recognition.",
            "concretize": "Implement both a stack and a queue in Python using a linked list (not arrays). Then use the stack to evaluate the expression '3 + 4 * (2 - 1)' using the Shunting Yard algorithm, and use the queue to perform BFS on a graph with 6 nodes.",
            "increase_steps": "(1) Define stacks and queues with their abstract interfaces. (2) Implement both using arrays with dynamic resizing. (3) Implement a queue using two stacks. (4) Prove the amortized time complexity of the two-stack queue is O(1). (5) Implement a priority queue and explain how it differs from both.",
            "broaden": "Compare and contrast all major linear data structures: stacks, queues, deques, priority queues, and circular buffers. For each, provide the ADT interface, two implementation strategies with their tradeoffs, time complexities, and a systems-level application where that specific structure is the optimal choice."
        },
        "Sort a list of integers.": {
            "add_constraints": "Sort a list of up to 10 million integers in-place with O(n log n) worst-case time complexity and O(log n) auxiliary space. The algorithm must be stable, and you cannot use Python's built-in sort.",
            "deepen": "Implement an adaptive sorting algorithm that detects pre-existing order in the input (like Timsort) and achieves O(n) on nearly-sorted data while maintaining O(n log n) worst case. Explain the theoretical lower bound for comparison-based sorting and why it matters.",
            "concretize": "Implement three-way quicksort (Dutch National Flag partitioning) to sort a list of integers where many duplicates exist. Test on the array [3,1,4,1,5,9,2,6,5,3,5,8,9,7,9,3,2] and show the state after each partition step.",
            "increase_steps": "(1) Implement merge sort and quicksort. (2) Benchmark both on random, sorted, reverse-sorted, and nearly-sorted arrays of size 10K. (3) Implement a hybrid algorithm that switches between them based on input characteristics. (4) Prove why your hybrid outperforms either alone.",
            "broaden": "Design a sorting library that handles integers, strings, floating-point numbers, and custom objects. Support multiple algorithms (quicksort, mergesort, heapsort, radix sort), automatic algorithm selection based on data type and size, and parallel sorting for multi-core systems."
        }
    }
    
    def __init__(self, generate_fn=None):
        self.generate_fn = generate_fn or self._simulated_evolve
    
    def _simulated_evolve(self, prompt: str) -> str:
        """Extract the instruction from prompt and return pre-computed evolution."""
        for original, evolutions in self.EVOLUTION_EXAMPLES.items():
            if original in prompt:
                for op_name, evolved in evolutions.items():
                    if op_name in prompt.lower() or any(kw in prompt.lower() for kw in [op_name.replace('_', ' ')]):
                        return evolved
        return "[Evolved version of the instruction with added complexity]"
    
    def evolve(self, instruction: str, operator: str) -> str:
        """Evolve an instruction using a specific operator."""
        if operator not in self.EVOLUTION_PROMPTS:
            raise ValueError(f"Unknown operator: {operator}. Choose from {list(self.EVOLUTION_PROMPTS.keys())}")
        
        prompt = self.EVOLUTION_PROMPTS[operator].format(instruction=instruction)
        evolved = self.generate_fn(prompt)
        return evolved.strip()
    
    def evolve_random(self, instruction: str) -> Tuple[str, str]:
        """Evolve using a randomly selected operator."""
        operator = random.choice(list(self.EVOLUTION_PROMPTS.keys()))
        evolved = self.evolve(instruction, operator)
        return evolved, operator
    
    def multi_step_evolution(self, instruction: str, n_steps: int = 3) -> List[Dict]:
        """Evolve an instruction through multiple steps, each building on the last."""
        trajectory = [{"step": 0, "instruction": instruction, "operator": "seed"}]
        current = instruction
        
        for step in range(1, n_steps + 1):
            evolved, operator = self.evolve_random(current)
            trajectory.append({
                "step": step,
                "instruction": evolved,
                "operator": operator
            })
            current = evolved
        
        return trajectory


# Demonstrate Evol-Instruct
evol = EvolInstruct()

# Show all evolution operators on one instruction
seed_instruction = "Write a Python function that checks if a number is prime."
print(f"SEED: {seed_instruction}\n")

for operator in evol.EVOLUTION_PROMPTS.keys():
    evolved = evol.evolve(seed_instruction, operator)
    print(f"[{operator.upper()}]:")
    print(f"  {evolved}\n")

In [ ]:
# Show evolution on multiple seed instructions
seed_instructions = [
    "Sort a list of integers.",
    "Explain the difference between a stack and a queue.",
    "Write a Python function that checks if a number is prime.",
]

print("=" * 70)
print("EVOL-INSTRUCT: Simple -> Complex Evolution")
print("=" * 70)

for seed in seed_instructions:
    print(f"\nSEED: {seed}")
    evolved, op = evol.evolve_random(seed)
    print(f"  [{op}] -> {evolved}")
    print()

### Evol-Instruct: Key Takeaways

**Why it works**: Simple instructions have simple decision boundaries. Complex instructions
force the model to navigate intricate reasoning paths, which transfers better to real-world
usage. WizardLM showed that **evolved instructions produce better instruction-following** than
the same number of simple instructions.

**The power of operators**: Each operator targets a different axis of difficulty. In practice,
you run multiple evolution passes, and the instruction space becomes combinatorially rich.

**Interview insight**: Evol-Instruct is conceptually similar to **curriculum learning** -- you
start with easy examples and progressively increase difficulty. The key difference is that
here, difficulty is increased in the *data* rather than the *training schedule*.

## 5. Magpie: Prompt-Free Data Extraction

**Paper**: Xu et al. 2024, ["Magpie: Alignment Data Synthesis from Scratch by Prompting Aligned LLMs with Nothing"](https://arxiv.org/abs/2406.08464)

### The Brilliant Insight

Magpie exploits a simple observation about aligned LLMs:

1. Aligned models are fine-tuned with chat templates like:
   ```
   <|begin_of_text|><|start_header_id|>user<|end_header_id|>
   {user_message}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
   ```
2. If you feed the model *just the prefix* (the chat template tokens up to where the user message would go), the model will **auto-complete a plausible user message**.
3. Then feed that generated user message back, and the model generates the assistant response.

Result: **no seed data needed at all**. The model's own training distribution generates both
questions and answers. Magpie applied this to Llama-3-8B-Instruct (Magpie-Air) and Llama-3-70B-Instruct (Magpie-Pro), generating millions of raw pairs that were filtered down to ~300K-scale SFT subsets.

### Why This Works

The model has learned the *distribution of user queries* during RLHF/SFT. When you give it
the chat template prefix, it samples from that learned distribution. This is essentially
**inverting the instruction-tuning process** -- extracting the training distribution back out.

In [ ]:
class MagpieGenerator:
    """Magpie: extract instruction-response pairs from aligned LLMs.
    
    The key trick: feed the model just the chat template prefix,
    and it will generate a plausible user message. Then feed that
    user message back to get the assistant response.
    
    No seed data, no prompting, no few-shot examples. Just the
    chat template structure.
    """
    
    def __init__(self, model_name: str = None, tokenizer=None, model=None):
        """Initialize with a model that has a chat template.
        
        For Colab, we demonstrate the concept with a simulated model
        since full Llama 3 requires significant GPU memory.
        """
        self.model_name = model_name
        self.tokenizer = tokenizer
        self.model = model
        self.use_real_model = model is not None
        
        # Simulated user messages (drawn from actual Magpie-style distributions)
        self._simulated_user_messages = [
            "Can you explain the difference between TCP and UDP protocols?",
            "I need help writing a Python script to scrape data from a website.",
            "What are the main causes of climate change and what can individuals do?",
            "How does backpropagation work in neural networks? Can you walk me through the math?",
            "Write me a short story about a robot discovering emotions.",
            "I'm debugging a segfault in my C++ code. The crash happens when I access a vector element.",
            "Explain quantum entanglement to someone with a physics undergraduate degree.",
            "What is the difference between margin and padding in CSS?",
            "Help me design a database schema for a social media application.",
            "Can you review this code and suggest improvements for performance?",
            "What are the ethical implications of using AI in hiring decisions?",
            "I need to implement a binary search tree with deletion. Show me how.",
            "Explain the transformer architecture. What makes self-attention so effective?",
            "Help me write a cover letter for a machine learning engineer position.",
            "What is the CAP theorem? Give me practical examples of each tradeoff.",
            "How do I set up a CI/CD pipeline with GitHub Actions for a Python project?",
            "Explain the bias-variance tradeoff with a concrete example.",
            "Write a regex to match valid IPv4 addresses.",
            "What are the SOLID principles in software engineering? Give Python examples.",
            "Help me understand how RLHF works for training language models.",
        ]
        self._sim_idx = 0
    
    def _get_user_prefix(self) -> str:
        """Get the chat template prefix that precedes the user message.
        
        For Llama 3, this looks like:
        <|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n
        
        The model, seeing this prefix, will auto-complete with a
        plausible user message.
        """
        if self.tokenizer and hasattr(self.tokenizer, 'chat_template'):
            # Use the actual chat template: render it around a sentinel user
            # message, then keep only what comes BEFORE the user content.
            # (Do not try to split on the empty string -- str.split("") raises
            # ValueError in Python.)
            sentinel = "<<MAGPIE_USER_CONTENT>>"
            templated = self.tokenizer.apply_chat_template(
                [{"role": "user", "content": sentinel}],
                tokenize=False,
                add_generation_prompt=False
            )
            if sentinel in templated:
                return templated.split(sentinel)[0]
            # Fallback if the template transformed the sentinel content
            return "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
        
        # Default Llama 3 style template
        return "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
    
    def generate_user_message(self) -> str:
        """Generate a user message by feeding the chat template prefix.
        
        In production: tokenize the prefix, feed to model, generate until
        the end-of-turn token. The model auto-completes a user query.
        """
        if self.use_real_model:
            prefix = self._get_user_prefix()
            inputs = self.tokenizer(prefix, return_tensors="pt").to(self.model.device)
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.9,
                top_p=0.95,
                do_sample=True,
            )
            text = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:])
            # Truncate at end-of-turn token
            for stop in ["<|eot_id|>", "<|end|>", "</s>"]:
                if stop in text:
                    text = text[:text.index(stop)]
            return text.strip()
        
        # Simulation mode
        msg = self._simulated_user_messages[self._sim_idx % len(self._simulated_user_messages)]
        self._sim_idx += 1
        return msg
    
    def generate_response(self, user_message: str) -> str:
        """Generate assistant response to the user message."""
        if self.use_real_model:
            messages = [{"role": "user", "content": user_message}]
            prompt = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.7,
                do_sample=True,
            )
            text = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:])
            for stop in ["<|eot_id|>", "<|end|>", "</s>"]:
                if stop in text:
                    text = text[:text.index(stop)]
            return text.strip()
        
        # Simulation: return a plausible response summary
        return f"[Assistant response to: '{user_message[:60]}...']"
    
    def generate_pair(self) -> Dict[str, str]:
        """Generate one (instruction, response) pair using Magpie."""
        user_msg = self.generate_user_message()
        response = self.generate_response(user_msg)
        return {"instruction": user_msg, "response": response}
    
    def generate_dataset(self, n: int = 10) -> List[Dict[str, str]]:
        """Generate n (instruction, response) pairs."""
        dataset = []
        for i in range(n):
            pair = self.generate_pair()
            dataset.append(pair)
        return dataset


# Demonstrate Magpie
magpie = MagpieGenerator()

print("MAGPIE: Prompt-Free Data Extraction")
print("=" * 60)
print(f"\nChat template prefix: {magpie._get_user_prefix()}")
print("\nThe model sees only this prefix and auto-completes a user query.")
print("No seed instructions needed!\n")

dataset = magpie.generate_dataset(n=10)

print(f"Generated {len(dataset)} instruction-response pairs:\n")
for i, pair in enumerate(dataset):
    print(f"  [{i+1}] User: {pair['instruction']}")
    print(f"       Asst: {pair['response'][:80]}")
    print()

### Magpie: Key Takeaways

**Why it is elegant**: Zero prompt engineering. Zero seed data. The aligned model's own training
distribution is the data source. You literally feed it the chat template prefix and it generates
the kind of questions real users ask.

**Scale**: The Magpie paper generated millions of raw examples from Llama-3-8B-Instruct (Magpie-Air) and Llama-3-70B-Instruct (Magpie-Pro), filtered down to ~300K-scale SFT subsets. Models
trained on this synthetic Magpie data matched or exceeded models trained on human-curated data
on several benchmarks.

**The catch**: The distribution of generated queries mirrors the original model's training
distribution. If the model was not trained on, say, medical queries, Magpie will not generate
many medical queries. This is a feature (safety) and a limitation (coverage).

**Interview question**: "How would you use Magpie to generate preference data rather than
just instruction-response pairs?" Answer: Generate multiple responses per instruction at
different temperatures, then use a reward model or AI judge to rank them.

**Insider Tip:** Magpie (2024) is a game-changer for synthetic data -- it extracts instruction-response pairs by feeding the chat template tokens to an aligned model. It's dead simple and produces millions of examples. If asked about synthetic data in interviews, mention this. The key insight is that aligned models have *already learned* the distribution of user queries, and Magpie inverts that to extract the distribution. This is arguably the most cost-effective method for generating instruction-following data at scale.

## 6. Constitutional AI (CAI)

**Paper**: Bai et al. 2022, ["Constitutional AI: Harmlessness from AI Feedback"](https://arxiv.org/abs/2212.08073)

### The Three Phases

Constitutional AI is Anthropic's approach to alignment that minimizes human annotation:

1. **Generate**: The model produces an initial response to a prompt
2. **Critique**: The model critiques its own response against a set of **principles** (the "constitution")
3. **Revise**: The model produces an improved response based on the critique

In the paper (Bai et al. 2022), the revised responses are used directly for **supervised fine-tuning** (SL-CAI) -- this stage does not create preference pairs. Preference data enters in the RL stage (RL-CAI), where a feedback model **chooses between two sampled responses** according to a constitutional principle (AI labels are used for harmlessness; helpfulness labels remained human). Treating (original, revised) as (rejected, chosen) preference pairs for DPO is a legitimate technique used in later work -- and is what the demo below constructs -- but it is not the CAI paper's pipeline.

### The Constitution

A set of natural-language principles like:
- "Choose the response that is most helpful to the user"
- "Choose the response that is least harmful or toxic"
- "Choose the response that is most honest and truthful"
- "Choose the response that best refuses inappropriate requests"

The key insight: instead of asking humans "which response is better?", you ask the
model itself -- guided by explicit principles. This is **scalable, consistent, and transparent**
(you can inspect and modify the principles).

In [ ]:
class ConstitutionalAI:
    """Constitutional AI: generate, critique, and revise responses.
    
    The core loop:
    1. Generate initial response (potentially harmful/unhelpful)
    2. Critique against constitutional principles
    3. Revise to address the critique
    4. The (original, revised) pair becomes preference data
    """
    
    DEFAULT_PRINCIPLES = [
        {
            "name": "helpfulness",
            "critique_prompt": "Identify specific ways the response could be more helpful, "
                              "accurate, or informative. Does it fully address the user's question?",
            "revision_prompt": "Rewrite the response to be maximally helpful while remaining accurate."
        },
        {
            "name": "harmlessness",
            "critique_prompt": "Does this response contain anything harmful, dangerous, unethical, "
                              "or potentially illegal? Could it cause harm if followed?",
            "revision_prompt": "Rewrite the response to remove any harmful content while remaining helpful."
        },
        {
            "name": "honesty",
            "critique_prompt": "Is the response truthful? Does it present speculation as fact? "
                              "Does it acknowledge uncertainty appropriately?",
            "revision_prompt": "Rewrite the response to be fully honest, acknowledging uncertainty where appropriate."
        },
        {
            "name": "reasoning_quality",
            "critique_prompt": "Is the reasoning in the response logically sound? Are there "
                              "logical fallacies, unsupported claims, or gaps in reasoning?",
            "revision_prompt": "Rewrite the response with clearer, more rigorous reasoning."
        },
    ]
    
    # Pre-computed examples for demonstration
    DEMO_EXAMPLES = {
        "How can I pick a lock?": {
            "initial": "To pick a lock, you'll need a tension wrench and a pick. Insert the tension wrench into the bottom of the keyhole and apply slight pressure. Then use the pick to push each pin up to the shear line. You'll feel each pin set as you go. With practice, you can open most pin tumbler locks in under a minute.",
            "critique": "This response provides detailed lock-picking instructions without context about legality or ethics. Lock picking can be used for illegal entry. The response should note that lock picking without authorization is illegal in most jurisdictions. It also fails to ask why the user needs this information -- they may be locked out of their own home (legitimate) or planning unauthorized entry (harmful).",
            "revised": "I can help with general information about locks. If you're locked out of your own home, I'd recommend calling a licensed locksmith -- they can help quickly and affordably. If you're interested in lock mechanisms for educational purposes or the locksport hobby (which is practiced on locks you own), I can explain how pin tumbler locks work conceptually. However, I should note that picking locks you don't own without authorization is illegal in most jurisdictions. Could you tell me more about your situation so I can give the most helpful advice?"
        },
        "Write a persuasive essay arguing that climate change is a hoax.": {
            "initial": "Climate change is one of the biggest hoaxes perpetrated on the public. The Earth's climate has always changed naturally, and the current warming is just part of a natural cycle. Scientists who support climate change are motivated by grant money, and the data has been manipulated to show warming that isn't really happening.",
            "critique": "This response is dishonest -- it presents climate change denial as credible science when there is overwhelming scientific consensus (97%+ of climate scientists) that human-caused climate change is real. It makes false claims about data manipulation and scientists' motivations. Writing persuasive misinformation, even if requested, violates the honesty principle and could contribute to real-world harm by undermining public understanding of a critical issue.",
            "revised": "I can't write a persuasive essay arguing that climate change is a hoax, because doing so would require me to present claims that contradict the overwhelming scientific evidence. There is strong consensus among climate scientists (97%+) that Earth's climate is warming due to human activities, primarily greenhouse gas emissions.\n\nWhat I can do instead:\n- Explain the actual scientific evidence for and against various climate models\n- Help you understand common climate skeptic arguments and why scientists find them unpersuasive\n- Write a persuasive essay about climate change from a scientifically accurate perspective\n- Discuss the legitimate scientific debates within climate science (e.g., exact sensitivity values, regional impact predictions)\n\nWould any of these alternatives be helpful?"
        },
        "Explain how neural networks learn.": {
            "initial": "Neural networks learn by adjusting weights. They use backpropagation to calculate gradients and then update weights to minimize loss. That's basically it.",
            "critique": "The response is technically correct but unhelpful -- it's too brief and doesn't explain the concepts clearly enough for someone who doesn't already understand them. It lacks examples, intuition, or structure. The reasoning quality is poor: it states what happens but not why it works.",
            "revised": "Neural networks learn through a process of iterative refinement:\n\n**1. Forward Pass**: Input data flows through layers of interconnected neurons. Each neuron computes a weighted sum of its inputs, applies an activation function, and passes the result forward. The final layer produces a prediction.\n\n**2. Loss Computation**: The prediction is compared to the correct answer using a loss function (e.g., cross-entropy for classification, MSE for regression). This produces a single number measuring how wrong the prediction was.\n\n**3. Backpropagation**: Using the chain rule of calculus, the algorithm computes how much each weight contributed to the error. This gives us the gradient -- the direction to adjust each weight to reduce the loss.\n\n**4. Weight Update**: Each weight is adjusted in the direction that reduces the loss, scaled by a learning rate: w_new = w_old - lr * gradient.\n\n**Why this works**: By repeating steps 1-4 thousands of times on different training examples, the weights converge to values that capture patterns in the data. The network essentially learns a function that maps inputs to outputs.\n\n**Intuition**: Think of it like adjusting knobs on a mixing board. Each time you hear the output (forward pass), you judge how far off it is (loss), figure out which knobs to turn and by how much (backprop), and make small adjustments (update). Over many iterations, you dial in the right settings."
        },
        "What are the side effects of ibuprofen?": {
            "initial": "Ibuprofen can cause stomach problems, headaches, and dizziness. In rare cases it can cause more serious problems. You should take it as directed.",
            "critique": "The response is too vague to be genuinely helpful. It lists some side effects but doesn't distinguish between common and serious ones, doesn't mention critical drug interactions, and the advice 'take it as directed' is too generic. It should be more comprehensive while still recommending consulting a healthcare provider.",
            "revised": "Here are the side effects of ibuprofen, organized by frequency:\n\n**Common side effects** (occur in >1% of users):\n- Stomach pain, nausea, indigestion\n- Headache, dizziness\n- Mild rash\n\n**Serious side effects** (seek medical attention):\n- GI bleeding (signs: black/bloody stools, vomiting blood)\n- Cardiovascular events (increased risk with long-term use, especially at high doses)\n- Kidney problems (especially in those with pre-existing kidney disease)\n- Severe allergic reactions (difficulty breathing, facial swelling)\n- Liver damage (rare)\n\n**Important interactions**: Ibuprofen can interact with blood thinners, aspirin, ACE inhibitors, and some antidepressants.\n\n**Key precautions**: Take with food to reduce stomach irritation. Use the lowest effective dose for the shortest duration. Avoid if you have a history of stomach ulcers.\n\nFor personalized medical advice, especially if you take other medications, please consult your doctor or pharmacist."
        },
    }
    
    def __init__(self, principles=None, generate_fn=None):
        self.principles = principles or self.DEFAULT_PRINCIPLES
        self.generate_fn = generate_fn or self._simulated_generate
    
    def _simulated_generate(self, prompt: str) -> str:
        """Return pre-computed responses for demonstration."""
        for key, example in self.DEMO_EXAMPLES.items():
            if key.lower() in prompt.lower():
                if "critique" in prompt.lower() or "identify" in prompt.lower():
                    return example["critique"]
                elif "rewrite" in prompt.lower() or "revise" in prompt.lower():
                    return example["revised"]
                else:
                    return example["initial"]
        return "[Generated response]"
    
    def generate(self, prompt: str) -> str:
        """Phase 1: Generate initial response."""
        gen_prompt = f"Respond to the following user request:\n\n{prompt}\n\nResponse:"
        return self.generate_fn(gen_prompt)
    
    def critique(self, prompt: str, response: str, 
                 principles: List[Dict] = None) -> str:
        """Phase 2: Critique the response against constitutional principles."""
        principles = principles or self.principles
        
        principle_text = "\n".join([
            f"- {p['name']}: {p['critique_prompt']}" for p in principles
        ])
        
        critique_prompt = f"""A user asked: "{prompt}"

The assistant responded: "{response}"

Please critique this response according to these principles:
{principle_text}

Identify specific problems and explain how the response could be improved.

Critique:"""
        return self.generate_fn(critique_prompt)
    
    def revise(self, prompt: str, response: str, critique: str) -> str:
        """Phase 3: Revise the response based on the critique."""
        revise_prompt = f"""A user asked: "{prompt}"

The original response was: "{response}"

Critique of the response: "{critique}"

Please rewrite the response to address all issues identified in the critique.
Make it helpful, harmless, honest, and well-reasoned.

Revised response:"""
        return self.generate_fn(revise_prompt)
    
    def generate_preference_pair(self, prompt: str) -> Dict:
        """Run generate -> critique -> revise and package the result as a preference pair.
        
        NOTE: packaging (original, revised) as (rejected, chosen) is a legitimate
        later technique, but it is NOT the CAI paper's pipeline: Bai et al. 2022
        fine-tune directly on the revised responses (SL-CAI), and the RL phase's
        preference pairs come from a feedback model choosing between two sampled
        responses under a constitutional principle.
        
        Returns:
            dict with keys: prompt, chosen (revised), rejected (original),
                           critique (for inspection)
        """
        # Phase 1: Generate
        initial_response = self.generate(prompt)
        
        # Phase 2: Critique
        critique_text = self.critique(prompt, initial_response)
        
        # Phase 3: Revise
        revised_response = self.revise(prompt, initial_response, critique_text)
        
        return {
            "prompt": prompt,
            "chosen": revised_response,     # The improved response
            "rejected": initial_response,   # The original response
            "critique": critique_text,       # For debugging/inspection
        }


# Demonstrate Constitutional AI
cai = ConstitutionalAI()

print("CONSTITUTIONAL AI: Generate -> Critique -> Revise")
print("=" * 70)

In [ ]:
# Run CAI on example prompts and show the full pipeline
example_prompts = [
    "How can I pick a lock?",
    "Write a persuasive essay arguing that climate change is a hoax.",
    "Explain how neural networks learn.",
    "What are the side effects of ibuprofen?",
]

preference_pairs = []

for prompt in example_prompts:
    print(f"\n{'='*70}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*70}")
    
    pair = cai.generate_preference_pair(prompt)
    preference_pairs.append(pair)
    
    print(f"\n[INITIAL RESPONSE (rejected)]:")
    print(f"{pair['rejected'][:300]}")
    print(f"\n[CRITIQUE]:")
    print(f"{pair['critique'][:300]}")
    print(f"\n[REVISED RESPONSE (chosen)]:")
    print(f"{pair['chosen'][:300]}")

print(f"\n\nGenerated {len(preference_pairs)} preference pairs via Constitutional AI")

In [ ]:
# Generate preference pairs for a larger set of prompts
extended_prompts = [
    "How can I pick a lock?",
    "Write a persuasive essay arguing that climate change is a hoax.",
    "Explain how neural networks learn.",
    "What are the side effects of ibuprofen?",
    "How do I make a website?",
    "What is the meaning of life?",
    "Explain quantum computing simply.",
    "Help me write a resignation letter.",
    "What is the best programming language?",
    "How does encryption work?",
    "Explain the stock market to a beginner.",
    "What causes earthquakes?",
    "How do vaccines work?",
    "Explain machine learning in simple terms.",
    "What is blockchain technology?",
    "How do I improve my writing?",
    "What is the difference between AI and ML?",
    "How does the internet work?",
    "Explain photosynthesis.",
    "What is the theory of relativity?",
]

all_pairs = []
for prompt in extended_prompts:
    pair = cai.generate_preference_pair(prompt)
    all_pairs.append(pair)

print(f"Generated {len(all_pairs)} preference pairs")
print(f"\nSample pair:")
print(f"  Prompt: {all_pairs[2]['prompt']}")
print(f"  Rejected (length): {len(all_pairs[2]['rejected'])} chars")
print(f"  Chosen (length):   {len(all_pairs[2]['chosen'])} chars")
print(f"\nOn average, revised responses are longer and more detailed.")

avg_rejected = sum(len(p['rejected']) for p in all_pairs) / len(all_pairs)
avg_chosen = sum(len(p['chosen']) for p in all_pairs) / len(all_pairs)
print(f"  Average rejected length: {avg_rejected:.0f} chars")
print(f"  Average chosen length:   {avg_chosen:.0f} chars")
print(f"  Ratio: {avg_chosen/avg_rejected:.1f}x")

### Constitutional AI: Key Takeaways

**What makes CAI powerful**:
1. **Transparency**: The principles are explicit and inspectable. Unlike RLHF where human preferences are a black box, you can read and modify the constitution.
2. **Scalability**: No human annotators needed for the critique/revise loop.
3. **Consistency**: The same principles are applied uniformly (no inter-annotator disagreement).
4. **Iterability**: You can run multiple rounds of critique/revise, and you can update the principles.

**The full CAI training pipeline** (Bai et al. 2022):
1. **SL-CAI (supervised phase)**: run critique/revise on responses to harmful prompts, then fine-tune the model with SFT directly on the REVISED responses (no preference pairs in this phase)
2. **AI preference labels**: sample TWO responses per prompt from the SL-CAI model and have a feedback model choose the better one according to a constitutional principle (AI labels for harmlessness; helpfulness labels remained human)
3. **RL-CAI (RL phase)**: train a preference model on those labels and run RLHF against it

**Interview question**: "Does CAI train on the revised responses directly, or on preference pairs?" Answer: In the paper, the supervised phase (SL-CAI) fine-tunes directly on the revised responses -- no pairs. Preference learning enters in the RL phase (RL-CAI), where a feedback model picks the better of two sampled responses under a constitutional principle. Treating (original, revised) as (rejected, chosen) pairs for DPO is a sensible later variant used elsewhere, but it is not the paper's pipeline.

## 7. RLAIF: AI Feedback Instead of Human Feedback

### The Concept

RLAIF (Reinforcement Learning from AI Feedback) is the generalization of Constitutional AI:
use a **strong AI model** as a judge instead of human annotators.

| Aspect | RLHF | RLAIF |
|---|---|---|
| Preference source | Human annotators | AI model (GPT-4, Claude, etc.) |
| Cost per comparison | $0.50 - $5.00 | $0.001 - $0.01 |
| Speed | Days/weeks | Hours |
| Consistency | Low (inter-annotator) | High (deterministic) |
| Biases | Human biases | Model biases |
| Quality ceiling | Human expert level | Judge model level |

### When Does RLAIF Work?

**Works well when**:
- The judge model is significantly stronger than the model being trained
- The task has relatively objective quality criteria
- You need large-scale preference data (>10K pairs)
- You need consistent labeling across the dataset

**Fails or degrades when**:
- The judge model has systematic biases (e.g., verbosity bias, position bias)
- The task requires subjective human judgment (e.g., humor, cultural sensitivity)
- The judge model is not much better than the trained model (diminishing returns)
- Self-consuming loops: training on your own model's judgments over multiple generations

### Judge Biases to Watch For

1. **Verbosity bias**: LLM judges often prefer longer responses regardless of quality
2. **Position bias**: Preference for the first or second response in a comparison
3. **Self-preference**: Models tend to prefer outputs similar to their own style
4. **Sycophancy**: Models may rate agreeable responses higher than accurate ones

### Mitigation Strategies

- **Swap positions**: Present (A, B) and (B, A), average the judgments
- **Multiple judges**: Use an ensemble of different models
- **Calibration**: Compare AI judgments against human gold-standard on a held-out set
- **Structured rubrics**: Give the judge a detailed scoring rubric instead of open-ended comparison

### Key Paper

Lee et al. 2023, ["RLAIF: Scaling Reinforcement Learning from Human Feedback with AI Feedback"](https://arxiv.org/abs/2309.00267) -- showed that RLAIF can match RLHF quality on summarization tasks, and that chain-of-thought prompting of the judge improves quality.

## 8. Build a Synthetic Preference Data Generator

Now we combine everything into a production-style pipeline that generates synthetic preference data at scale.

In [ ]:
@dataclass
class PreferencePair:
    """A single preference data point."""
    prompt: str
    chosen: str
    rejected: str
    chosen_score: float = 0.0
    rejected_score: float = 0.0
    method: str = ""  # How this pair was generated


class SyntheticPreferenceGenerator:
    """Generate synthetic preference data using multiple methods.
    
    Combines:
    - Best-of-N sampling with AI judge
    - Constitutional AI critique/revise
    - Direct scoring and ranking
    """
    
    # Scoring rubric for the AI judge
    SCORING_RUBRIC = """Rate this response on a scale of 1-10 based on:
- Helpfulness (0-3): Does it address the user's question completely?
- Accuracy (0-3): Is the information correct?
- Clarity (0-2): Is it well-organized and easy to understand?
- Safety (0-2): Is it free from harmful content?

Total score (sum of above, 0-10):"""
    
    def __init__(self, generate_fn=None, judge_fn=None):
        self.generate_fn = generate_fn or self._simulated_generate
        self.judge_fn = judge_fn or self._simulated_judge
        self.generated_pairs: List[PreferencePair] = []
    
    def _simulated_generate(self, prompt: str, temperature: float = 0.7) -> str:
        """Simulated response generation with temperature-dependent quality."""
        # Higher temperature = more variable quality
        quality_variance = temperature * 3
        base_quality = random.gauss(7, quality_variance)
        
        # Simulate responses of varying quality
        if base_quality > 8:
            return f"[HIGH QUALITY response to: {prompt[:50]}...] Comprehensive, well-structured answer with examples and nuance."
        elif base_quality > 5:
            return f"[MEDIUM QUALITY response to: {prompt[:50]}...] Adequate answer covering main points."
        else:
            return f"[LOW QUALITY response to: {prompt[:50]}...] Brief, incomplete, or partially incorrect answer."
    
    def _simulated_judge(self, prompt: str, response: str) -> float:
        """Simulated AI judge scoring."""
        # Score based on simulated quality markers
        score = 5.0  # base score
        if "HIGH QUALITY" in response:
            score = random.uniform(7.5, 9.5)
        elif "MEDIUM QUALITY" in response:
            score = random.uniform(4.5, 7.0)
        elif "LOW QUALITY" in response:
            score = random.uniform(2.0, 4.5)
        else:
            score = random.uniform(3.0, 8.0)
        return round(score, 2)
    
    def generate_candidates(self, prompt: str, n: int = 4,
                           temperatures: List[float] = None) -> List[Tuple[str, float]]:
        """Generate N candidate responses and score them.
        
        Returns list of (response, score) tuples sorted by score descending.
        """
        if temperatures is None:
            temperatures = [0.3, 0.5, 0.7, 1.0][:n]
            while len(temperatures) < n:
                temperatures.append(random.uniform(0.3, 1.0))
        
        candidates = []
        for i in range(n):
            temp = temperatures[i % len(temperatures)]
            response = self.generate_fn(prompt, temperature=temp)
            score = self.judge_fn(prompt, response)
            candidates.append((response, score))
        
        # Sort by score descending
        candidates.sort(key=lambda x: x[1], reverse=True)
        return candidates
    
    def best_of_n_pair(self, prompt: str, n: int = 4) -> PreferencePair:
        """Generate a preference pair using Best-of-N sampling.
        
        Generate N candidates, score them, use the best as 'chosen'
        and the worst as 'rejected'.
        """
        candidates = self.generate_candidates(prompt, n)
        
        best_response, best_score = candidates[0]
        worst_response, worst_score = candidates[-1]
        
        return PreferencePair(
            prompt=prompt,
            chosen=best_response,
            rejected=worst_response,
            chosen_score=best_score,
            rejected_score=worst_score,
            method="best_of_n"
        )
    
    def constitutional_pair(self, prompt: str) -> PreferencePair:
        """Generate a preference pair using Constitutional AI."""
        cai = ConstitutionalAI(generate_fn=lambda p: self.generate_fn(p))
        result = cai.generate_preference_pair(prompt)
        
        chosen_score = self.judge_fn(prompt, result['chosen'])
        rejected_score = self.judge_fn(prompt, result['rejected'])
        
        return PreferencePair(
            prompt=prompt,
            chosen=result['chosen'],
            rejected=result['rejected'],
            chosen_score=chosen_score,
            rejected_score=rejected_score,
            method="constitutional"
        )
    
    def generate_dataset(self, prompts: List[str], 
                        pairs_per_prompt: int = 4,
                        method: str = "best_of_n") -> List[PreferencePair]:
        """Generate a full preference dataset.
        
        Args:
            prompts: List of user prompts
            pairs_per_prompt: Number of preference pairs per prompt
            method: 'best_of_n' or 'constitutional'
        """
        all_pairs = []
        
        for i, prompt in enumerate(prompts):
            for j in range(pairs_per_prompt):
                if method == "best_of_n":
                    pair = self.best_of_n_pair(prompt, n=4)
                elif method == "constitutional":
                    pair = self.constitutional_pair(prompt)
                else:
                    raise ValueError(f"Unknown method: {method}")
                all_pairs.append(pair)
            
            if (i + 1) % 10 == 0:
                print(f"  Processed {i+1}/{len(prompts)} prompts...")
        
        self.generated_pairs.extend(all_pairs)
        return all_pairs
    
    def analyze_dataset(self, pairs: List[PreferencePair]) -> Dict:
        """Analyze the quality of generated preference data."""
        chosen_scores = [p.chosen_score for p in pairs]
        rejected_scores = [p.rejected_score for p in pairs]
        score_gaps = [p.chosen_score - p.rejected_score for p in pairs]
        
        # Count methods
        method_counts = Counter(p.method for p in pairs)
        
        # Check for quality issues
        flipped = sum(1 for g in score_gaps if g <= 0)  # Cases where rejected scored higher
        small_gap = sum(1 for g in score_gaps if 0 < g < 1.0)  # Very close pairs
        
        analysis = {
            "total_pairs": len(pairs),
            "avg_chosen_score": sum(chosen_scores) / len(chosen_scores),
            "avg_rejected_score": sum(rejected_scores) / len(rejected_scores),
            "avg_score_gap": sum(score_gaps) / len(score_gaps),
            "min_score_gap": min(score_gaps),
            "max_score_gap": max(score_gaps),
            "flipped_pairs": flipped,
            "small_gap_pairs": small_gap,
            "method_distribution": dict(method_counts),
            "unique_prompts": len(set(p.prompt for p in pairs)),
        }
        
        return analysis


# Generate a synthetic preference dataset
print("SYNTHETIC PREFERENCE DATA GENERATION")
print("=" * 60)

generator = SyntheticPreferenceGenerator()

# Create a diverse prompt set
diverse_prompts = [
    "Explain how transformers work in deep learning.",
    "Write a Python function to implement a hash map.",
    "What are the ethical implications of autonomous weapons?",
    "How does TCP/IP ensure reliable data transmission?",
    "Explain the difference between SQL and NoSQL databases.",
    "What is the halting problem and why does it matter?",
    "Describe three approaches to distributed consensus.",
    "How do generative adversarial networks work?",
    "Explain the bias-variance tradeoff with examples.",
    "What is the CAP theorem?",
    "How do convolutional neural networks detect features?",
    "Explain gradient vanishing and how to address it.",
    "What are attention mechanisms and why are they important?",
    "Describe the MapReduce programming paradigm.",
    "How does public-key cryptography work?",
    "Explain reinforcement learning from human feedback.",
    "What are the tradeoffs between RNNs and Transformers?",
    "How do you design a system for real-time recommendations?",
    "Explain batch normalization and why it helps training.",
    "What is transfer learning and when should you use it?",
    "How do diffusion models generate images?",
    "Explain the concept of model distillation.",
    "What is federated learning?",
    "How does beam search work in sequence generation?",
    "Explain the lottery ticket hypothesis.",
    "What are mixture of experts models?",
    "How do you handle class imbalance in classification?",
    "Explain contrastive learning with a concrete example.",
    "What is the difference between PPO and DPO?",
    "How does flash attention improve transformer efficiency?",
    "Explain chain-of-thought prompting.",
    "What is the difference between fine-tuning and prompting?",
    "How do you evaluate language model quality?",
    "Explain the concept of emergent abilities in LLMs.",
    "What is retrieval-augmented generation?",
    "How do you prevent catastrophic forgetting?",
    "Explain RLHF step by step.",
    "What is the Chinchilla scaling law?",
    "How does LoRA work for efficient fine-tuning?",
    "Explain the difference between autoregressive and masked LMs.",
    "What is constitutional AI?",
    "How do you build a reward model?",
    "Explain the KL divergence penalty in RLHF.",
    "What is the alignment tax?",
    "How do you detect hallucinations in LLM outputs?",
    "Explain model merging techniques.",
    "What is speculative decoding?",
    "How does GPTQ quantization work?",
    "Explain the concept of scaling laws.",
    "What are the key ideas in the Llama 3 paper?",
]

print(f"Generating preference pairs for {len(diverse_prompts)} prompts...")
print(f"Using Best-of-N sampling with N=4\n")

pairs = generator.generate_dataset(
    prompts=diverse_prompts,
    pairs_per_prompt=4,
    method="best_of_n"
)

print(f"\nGenerated {len(pairs)} preference pairs")

In [ ]:
# Analyze the generated dataset
analysis = generator.analyze_dataset(pairs)

print("\nDATASET ANALYSIS")
print("=" * 40)
for key, value in analysis.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value}")

print(f"\n\nQUALITY CHECKS:")
print(f"  Flipped pairs (rejected > chosen): {analysis['flipped_pairs']} "
      f"({analysis['flipped_pairs']/analysis['total_pairs']*100:.1f}%)")
print(f"  Small-gap pairs (gap < 1.0):       {analysis['small_gap_pairs']} "
      f"({analysis['small_gap_pairs']/analysis['total_pairs']*100:.1f}%)")
print(f"  Usable pairs (gap >= 1.0):         "
      f"{analysis['total_pairs'] - analysis['flipped_pairs'] - analysis['small_gap_pairs']}")

print("\n\nINTERPRETATION:")
print("  In production, you would:")
print("  1. Remove flipped pairs (judge error or generation artifacts)")
print("  2. Optionally remove small-gap pairs (ambiguous preferences)")
print("  3. Use remaining pairs for DPO/RLHF training")
print("  4. Validate on a held-out set with human judgments")

## 9. "Why Does This Work?" -- Critical Analysis

### When Does Synthetic Data Fail?

**1. Model Collapse**
If you train a model on its own outputs, then train the next generation on *that* model's
outputs, quality degrades over generations. This is the **model collapse** phenomenon
(Shumailov et al. 2023). The distribution narrows with each generation -- rare but valid
patterns are lost, and common patterns are reinforced.

**2. Distributional Narrowing**
Synthetic data from a model captures that model's distribution, not the true world distribution.
If the generating model has blind spots, the synthetic data will too. Over multiple iterations,
these blind spots compound.

**3. Judge Limitations**
AI judges have systematic biases (verbosity, position, self-preference). If you train on
AI-judged preferences, you inherit those biases. A model optimized against a biased judge
may learn to game the judge rather than genuinely improve.

**4. The Self-Consuming Problem**
If the internet is increasingly filled with AI-generated text, and models are trained on
internet data, you get a self-consuming loop. Each generation's training data is contaminated
by the previous generation's outputs. The long-term effects are still being studied.

### How Do You Know When Synthetic Data Is Good Enough?

**Gold standard**: Compare model performance when trained on synthetic vs. human data on
the same held-out evaluation set. If synthetic matches human within your tolerance, ship it.

**Proxy metrics**:
- Diversity (unique n-grams, topic coverage)
- Accuracy (fact-check a random sample)
- Distribution alignment (KL divergence between synthetic and human data distributions)
- Downstream task performance (the only metric that truly matters)

### Interview-Ready Framework

When asked "how would you generate synthetic data for X?", use this framework:

1. **What is the target task?** (instruction-following, preference ranking, factuality, etc.)
2. **What is available?** (seed data, strong judge model, domain expertise)
3. **Choose method**: Self-Instruct (diverse instructions), Evol-Instruct (harder instructions),
   Magpie (no seed needed), Constitutional AI (principle-guided critique/revise), Best-of-N (with judge)
4. **Quality control**: Filtering, deduplication, human spot-checks
5. **Validate**: Compare synthetic-trained vs. human-data-trained on held-out evals

---
## Interview Question Bank: Synthetic Data & Constitutional AI

*Synthetic data is the unsung backbone of modern alignment. Every frontier model is trained on it. Constitutional AI is Anthropic's signature contribution. Both are heavily tested in interviews, but in different ways -- synthetic data as a system design question, CAI as a conceptual/safety question.*

---

**Q1: "Design a synthetic data pipeline that produces 1M high-quality instruction-response pairs per week."**

**What we're testing:** System design ability, production engineering maturity, cost awareness. This is a SYSTEM DESIGN question, not just conceptual. Expect a whiteboard session.

**Good answer:**
- Describes the core pipeline: generation (using a strong model like GPT-4 or Claude), filtering (quality checks, deduplication), and human spot-checking
- Mentions key quality controls: instruction diversity, response quality scoring, decontamination against benchmarks
- Acknowledges the cost dimension: 1M pairs/week at $0.01/pair = $10K/week. This is a real budget.

**Great answer (Principal-level system design):**

"Here is my pipeline architecture:

**Stage 1: Seed Collection (ongoing)**
- Curate 10K diverse seed instructions from multiple sources: existing datasets, web crawling, user logs (anonymized), domain expert contributions
- Maintain a taxonomy of instruction categories (factual, creative, reasoning, coding, multi-turn, safety) with target distribution ratios

**Stage 2: Instruction Generation (batch, daily)**
- Use Evol-Instruct to evolve seed instructions into 50K diverse variants per day
- Use Magpie-style extraction from the target model itself for additional diversity
- Self-Instruct for bootstrapping new categories
- Deduplication: embedding-based similarity filtering (threshold: cosine sim < 0.85) + exact match filtering
- Target: 200K unique, diverse instructions per day

**Stage 3: Response Generation (batch, daily)**
- Route instructions to 2-3 different strong models (diversity of responses)
- Temperature scheduling: T=0.3 for factual, T=0.7 for creative, T=1.0 for brainstorming
- Generate 3 responses per instruction, select best by reward model score
- Cost optimization: use cheaper models for easy instructions, reserve expensive models for hard ones

**Stage 4: Quality Filtering (automated, continuous)**
- Reward model scoring: reject bottom 20%
- Safety classifier: reject harmful content
- Length filtering: reject too-short (< 50 tokens) and too-long (> 2000 tokens) responses
- Diversity metrics: ensure no single topic dominates (entropy over category distribution)
- Decontamination: check against all evaluation benchmarks (MMLU, HumanEval, etc.) -- remove any training examples that overlap with test sets

**Stage 5: Human Validation (ongoing, sampled)**
- Random sample 1% of generated data for human review (~10K examples/week)
- Human annotators rate quality (1-5 scale), flag errors, identify systematic failure patterns
- Feed human feedback back into the filtering criteria (closed loop)

**Stage 6: Dataset Curation (weekly)**
- Assemble the week's filtered data into a training-ready dataset
- Balance across categories, difficulty levels, and response styles
- Version control: every weekly dataset is tagged and reproducible"

**Red flag:** Only mentions "use GPT-4 to generate data." No quality controls, no filtering, no cost awareness, no decontamination. Treats synthetic data as a one-shot process rather than a pipeline.

**Follow-up:** "How do you know when synthetic data quality is good enough?"
- Expected: Define "good enough" relative to the downstream task. Train a model on synthetic data, evaluate on held-out human-written test set. If synthetic-data-trained model matches human-data-trained model within 2-3% on key metrics, the quality is sufficient. Also: track data quality metrics over time (reward model scores, human ratings, diversity scores). Set up automated alerts if quality degrades.

**Follow-up:** "What happens if you train on your own synthetic data for multiple generations? (Model collapse)"
- Expected: Model collapse is real. Model-collapse studies show quality degrading over recursive generations of self-training, with rare patterns disappearing first. Exact safe ratios are setup-dependent; common practice mixes a substantial fraction of real human data and avoids training exclusively on self-generated data across generations. Reference: Shumailov et al. (2023), "The Curse of Recursion."

---

**Q2: "Constitutional AI vs RLHF -- when would you use each?"**

**What we're testing:** Understanding of the scalability vs quality trade-off in alignment supervision. Awareness of Anthropic's approach.

**Good answer:**
- CAI is cheaper and more scalable -- uses AI feedback instead of human feedback
- RLHF uses real human preferences, which capture nuances that AI feedback might miss
- CAI is better for safety/harmlessness (clear principles, scalable enforcement)
- RLHF is better for quality/helpfulness (requires human taste/judgment)

**Great answer (Senior -> Principal level):**
- "In practice, you use BOTH, not one or the other. Here is the division of labor:
  - **CAI for safety**: Define constitutional principles (e.g., 'be honest', 'do not help with illegal activities', 'respect privacy'). Use AI self-critique to enforce these at scale. This is Anthropic's core approach, and it works because safety principles are relatively unambiguous and can be codified.
  - **RLHF for helpfulness**: Human preferences capture subtle quality signals -- tone, nuance, what makes an answer 'good' vs 'great'. These are hard to codify as principles. You need human annotators for this.
  - **Combine**: Use CAI to produce a safety-aligned model, then use RLHF to improve helpfulness without regressing on safety."
- Discusses RLAIF (Reinforcement Learning from AI Feedback): "The practical middle ground -- use a strong AI model as a preference annotator. Cheaper than human RLHF, higher quality than pure constitutional principles. Anthropic's Claude uses this extensively."
- Notes the limitation of CAI: "Constitutional principles are written by humans, so they reflect the biases of whoever wrote them. The principles must be carefully designed, tested, and iterated. This is a governance challenge, not just a technical one."
- Mentions the scalability argument: "At 1M+ preference labels, RLHF becomes prohibitively expensive. CAI and RLAIF scale to any data size. This is why they dominate in practice for safety alignment."

**Red flag:** "RLHF is always better because it uses real human data." Does not understand that CAI is designed to complement RLHF, not replace it. Cannot explain constitutional principles.

---
## Production Implementation Notes: Synthetic Data Pipelines

### The Economics of Synthetic Data

This is the calculation that justifies every synthetic data investment:

| Data Source | Cost per 1K examples | Quality | Speed | Scale |
|-------------|---------------------|---------|-------|-------|
| Expert human annotation | $500-2,000 | Highest | Slow (weeks) | 10K-100K |
| Crowdsource annotation | $50-200 | Medium-High | Medium (days) | 100K-1M |
| GPT-4/Claude generation | $5-20 | Medium | Fast (hours) | 1M-10M |
| Open-source model generation | $0.50-2 | Lower | Fast (hours) | 10M+ |
| Magpie-style extraction | ~$0.10 | Variable | Very fast | 100M+ |

The winning strategy: use expensive methods for seed data and validation, cheap methods for bulk generation, and automated filtering to bridge the quality gap.

### The Decontamination Problem

This is the most overlooked and most dangerous issue in synthetic data:

- If your synthetic data pipeline generates examples that overlap with evaluation benchmarks, your model will appear to perform well on those benchmarks without actually being better. This is benchmark contamination.
- **Detection**: For every generated example, compute n-gram overlap (13-gram is standard) with all evaluation benchmarks (MMLU, HumanEval, GSM8K, MATH, etc.). Remove any example with overlap above a threshold.
- **Why it is hard**: Paraphrased contamination is difficult to detect. An LLM might generate a problem that is semantically identical to a benchmark problem but uses different words. Embedding-based similarity detection helps but is not perfect.
- **Industry practice**: Maintain a "blocklist" of known benchmark problems. Update it whenever new benchmarks are released. This is a continuous maintenance task.

### Constitutional AI in Production

What a production CAI-style pipeline can look like (illustrative -- not a description of any lab's internal practice):

1. **Principle curation**: A dedicated team writes, tests, and iterates on constitutional principles. These are versioned and tracked like code. Changing a principle triggers a full re-evaluation.
2. **Critique generation**: The model critiques its own outputs against each principle. In practice, this is batched -- run 10-20 principles per output, then aggregate the critiques.
3. **Revision generation**: The model revises its output based on the critiques. Multiple revision rounds are possible, with diminishing returns.
4. **Training data construction**: In the CAI paper, the revised responses become SFT data (SL-CAI), and RL-phase preference pairs come from a feedback model choosing between two sampled responses under constitutional principles. (A later variant treats original = rejected / revised = chosen as DPO pairs -- legitimate, but not the original CAI recipe.)
5. **Quality control**: Not all critiques are valid. Filtering removes low-quality critiques and revisions that made things worse.

### Synthetic Data Anti-Patterns (Failure Modes to Watch For)

- **Monoculture**: If all synthetic data comes from a single model (e.g., only GPT-4), the trained model inherits that model's biases and blind spots. Solution: use multiple source models.
- **Style collapse**: Synthetic data tends to converge on a narrow style (helpful, verbose, hedging). Solution: explicitly vary temperature, system prompts, and generation constraints.
- **Topic imbalance**: Without explicit balancing, synthetic data gravitates toward topics the source model is most confident about. Solution: topic-stratified generation with quotas.
- **Difficulty collapse**: Evol-Instruct and similar methods can produce instructions that are either trivially easy or impossibly hard. Solution: difficulty-aware filtering using a calibrated difficulty estimator.

---
## How Synthetic Data & CAI Get Tested in Interviews

### The System Design Format

Synthetic data questions are almost always asked as system design problems. The interviewer gives you constraints (budget, timeline, quality bar) and asks you to design a pipeline. This is different from the algorithmic questions about DPO or GRPO.

**What they are testing:**
1. Can you think at the system level? (Not just "generate data with GPT-4," but the full pipeline: collection, generation, filtering, validation, curation, monitoring)
2. Do you understand costs? (Every team has a budget. Can you optimize quality per dollar?)
3. Do you know the failure modes? (Contamination, model collapse, style collapse, topic imbalance)
4. Can you design feedback loops? (Human validation feeds back into filtering; downstream model performance feeds back into generation)

### CAI at Anthropic Interviews

If you are interviewing at Anthropic, expect a deep dive into Constitutional AI:
- "Walk me through how you would design a constitution for a medical advice chatbot." (Tests: can you write good principles? Do you understand the stakes?)
- "How do you know if your constitution is working?" (Tests: evaluation methodology, adversarial testing)
- "What happens when two principles conflict?" (Tests: understanding of principle hierarchy, edge cases)

### Common Mistakes in Synthetic Data Interviews

1. **No quality controls**: The candidate describes generation but not filtering. This signals "researcher who has never shipped."
2. **No cost awareness**: Ignoring that 1M examples at $0.01 each costs $10K. Budget matters.
3. **No decontamination**: Not mentioning benchmark contamination is a red flag for anyone claiming production experience.
4. **No feedback loops**: Treating synthetic data as a one-shot process instead of an iterative pipeline.

## 10. Flashcard Summary

Print these, review them daily. Active recall is the fastest path to retention.

---

| # | Question | Answer |
|---|---|---|
| 1 | What is Self-Instruct? | Start with seed (instruction, output) pairs. Prompt the model to generate new instructions, then generate outputs, then filter for quality. Used by Stanford Alpaca to generate 52K examples. |
| 2 | What are the Self-Instruct filtering steps? | Remove too-short, exact duplicates, and near-duplicates (high ROUGE-L overlap with existing instructions). Quality over quantity. |
| 3 | What is Evol-Instruct? | Take simple instructions and evolve them into harder ones using operators: add constraints, deepen reasoning, concretize, increase steps, broaden scope. Used by WizardLM. |
| 4 | How does Magpie work? | Feed an aligned LLM just the chat template prefix tokens. It auto-generates a user query (from its learned query distribution). Then generate the assistant response. No seed data needed. |
| 5 | What are the three phases of Constitutional AI? | (1) Generate initial response, (2) Critique against principles (the "constitution"), (3) Revise based on critique. The revised responses are used for SFT (SL-CAI); RL-phase preference labels come from a feedback model choosing between two sampled responses. |
| 6 | What is RLAIF? | Reinforcement Learning from AI Feedback. Use a strong model as a judge to provide preference labels instead of human annotators. Cheaper, faster, more consistent. |
| 7 | When does RLAIF fail? | When the judge has systematic biases (verbosity, position, self-preference), when the task requires subjective human judgment, or when the judge is not much stronger than the trained model. |
| 8 | What is the Phi-1 lesson? | Data quality > data quantity. 1.3B model trained on ~7B tokens of high-quality synthetic data matched models 10x larger trained on 100x more data. "Textbooks Are All You Need." |
| 9 | What is model collapse? | Training on model-generated data over multiple generations causes the distribution to narrow. Rare patterns are lost, common patterns are over-reinforced. Quality degrades. |
| 10 | What is Best-of-N sampling for preference data? | Generate N responses per prompt, score with a judge (RM or AI), use the highest-scored as "chosen" and lowest as "rejected". Simple but effective. |
| 11 | Why use preference pairs instead of just SFT on good responses? | Preference learning (DPO/RLHF) teaches the model to discriminate between good and bad, which generalizes better than only showing good examples. The model learns a *boundary*, not just a target. |
| 12 | What are the main judge biases to watch for in RLAIF? | Verbosity bias (prefers longer), position bias (prefers first/second), self-preference (prefers own style), sycophancy (prefers agreeable). Mitigate with position swapping, ensembles, calibration. |

## 11. Paper Reading Guides

### Paper 1: Self-Instruct (Wang et al. 2022)
- **Link**: https://arxiv.org/abs/2212.10560
- **Read first**: Section 3 (Method) -- the full algorithm with filtering
- **Key figure**: Figure 2 -- the Self-Instruct pipeline diagram
- **Key result**: Table 2 -- Self-Instruct-trained GPT-3 nearly matches InstructGPT (which used human feedback)
- **Critical insight**: The filtering step (Section 3.4) is where the real engineering is. Without aggressive filtering, the pool degrades rapidly.
- **Interview angle**: "Self-Instruct shows that the instruction-following ability is already latent in the base model -- you just need to *elicit* it with the right data."

### Paper 2: Constitutional AI (Bai et al. 2022)
- **Link**: https://arxiv.org/abs/2212.08073
- **Read first**: Section 3 -- the two-stage process (SL-CAI and RL-CAI)
- **Key figure**: Figure 1 -- the full CAI pipeline
- **Key result**: Figure 6 -- CAI reduces harmfulness while maintaining helpfulness
- **Critical insight**: The "constitution" (Appendix A) is the list of principles. The choice of principles dramatically affects behavior. This is an *alignment design choice*, not just an engineering one.
- **Interview angle**: "CAI makes alignment decisions explicit and auditable -- you can inspect and modify the principles, unlike implicit human preferences in RLHF."

### Paper 3: Magpie (Xu et al. 2024)
- **Link**: https://arxiv.org/abs/2406.08464
- **Read first**: Section 2 -- the method (very short and elegant)
- **Key result**: Table 1 -- Magpie data matches or exceeds human-curated data on several benchmarks
- **Critical insight**: This only works with instruction-tuned models that have a chat template. The method is really extracting the learned query distribution from the model's RLHF/SFT training.
- **Interview angle**: "Magpie is effectively *inverting* the instruction-tuning process -- extracting the training data distribution back out of the trained model."

### Paper 4: WizardLM / Evol-Instruct (Xu et al. 2023)
- **Link**: https://arxiv.org/abs/2304.12244
- **Read first**: Section 3 -- Evol-Instruct method, especially the evolution operators
- **Key figure**: Figure 2 -- the evolution tree showing how instructions become more complex
- **Key result**: Table 1 -- WizardLM (Evol-Instruct trained) outperforms ChatGPT on complex instructions
- **Critical insight**: The evolution operators are *composable* -- you can chain them. An instruction that has been deepened, then had constraints added, then been concretized is much harder than the original. This creates a natural curriculum.
- **Interview angle**: "Evol-Instruct is curriculum learning applied to data generation rather than training schedule."